In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"
os.environ["HF_HOME"] = "/local1/mohsenfayyaz/.hfcache/"

!git -C ColBERT/ pull || git clone https://github.com/stanford-futuredata/ColBERT.git
import sys; sys.path.insert(0, 'ColBERT/')

try: # When on google Colab, let's install all dependencies with pip.
    import google.colab
    !pip install -U pip
    !pip install -e ColBERT/['faiss-gpu','torch']
except Exception:
  import sys; sys.path.insert(0, 'ColBERT/')
  try:
    from colbert import Indexer, Searcher
  except Exception:
    print("If you're running outside Colab, please make sure you install ColBERT in conda following the instructions in our README. You can also install (as above) with pip but it may install slower or less stable faiss or torch dependencies. Conda is recommended.")
    assert False

Already up to date.


In [2]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df_foil = pd.read_json("hf://datasets/mohsenfayyaz/ColDeR/test/foil.jsonl", lines=True)
df_foil.head(1)

,query,document_1,document_2,head_entity_names,tail_entity_names,relation_name,relation,title,id,head_entity,tail_entity,head_entity_longest_name,tail_entity_longest_name,head_entity_types,tail_entity_types,evidence_sent_ids,evidence_sents,sents
0,When was The Private Life of Helen of Troy pub...,""" The Private Life of Helen of Troy "" "" The Pr...",The discography of English rock band Joy Divis...,[The Private Life of Helen of Troy],[1927],publication date,P577,The Private Life of Helen of Troy,validation13053,"[{'name': 'The Private Life of Helen of Troy',...","[{'type': 'TIME', 'pos': [9, 10], 'name': '192...",The Private Life of Helen of Troy,1927,[MISC],[TIME],[0],"[[The, Private, Life, of, Helen, of, Troy, is,...","[[The, Private, Life, of, Helen, of, Troy, is,..."


In [3]:
import torch
import numpy as np
from colbert.modeling.checkpoint import Checkpoint
from colbert.infra import ColBERTConfig
from colbert.modeling.colbert import colbert_score

checkpoint = 'colbert-ir/colbertv2.0'
config = ColBERTConfig(doc_maxlen=510, nbits=2)
ckpt = Checkpoint(checkpoint, colbert_config=config)

def get_colbert_scores(query, data):
    Q = ckpt.queryFromText([query])
    D = ckpt.docFromText(data, bsize=32)[0]
    D_mask = torch.ones(D.shape[:2], dtype=torch.long)
    scores = colbert_score(Q, D, D_mask).flatten().cpu().numpy().tolist()
    ranking = np.argsort(scores)[::-1]
    return scores, ranking

/data2/mohsenfayyaz/projects/Retriever-Contextualization/src/notebooks/Rebuttal/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


In [4]:
from tqdm.auto import tqdm

rdf = []
for row in tqdm(df_foil.to_dict(orient="records")):
    query = row["query"]
    data = [row["document_1"], row["document_2"]]
    scores, ranking = get_colbert_scores(query, data)
    rdf.append({"query": query, "document_1": data[0], "document_2": data[1], "score_1": scores[0], "score_2": scores[1], "ranking": ranking})
    
rdf = pd.DataFrame(rdf)
rdf

  0%|          | 0/250 [00:00<?, ?it/s]


#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: When was The Private Life of Helen of Troy published?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([ 101,    1, 2043, 2001, 1996, 2797, 2166, 1997, 6330, 1997, 9553, 2405,
        1029,  102,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



/data2/mohsenfayyaz/projects/Retriever-Contextualization/src/notebooks/Rebuttal/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


,query,document_1,document_2,score_1,score_2,ranking
0,When was The Private Life of Helen of Troy pub...,""" The Private Life of Helen of Troy "" "" The Pr...",The discography of English rock band Joy Divis...,23.890625,22.015625,"[0, 1]"
1,What is EMH an instance of?,""" EMH "" "" EMH "" Guest star Leland Orser plays ...",The University of Uyo ( UNIUYO ) is located in...,18.718750,12.843750,"[0, 1]"
2,Which record label is God 's Son associated with?,""" God 's Son "" "" God 's Son "" Partly inspired ...","The Metacomet Ridge , Metacomet Ridge Mountain...",18.593750,17.484375,"[0, 1]"
3,What is a notable work of Miami Sound Machine?,""" Miami Sound Machine "" "" Miami Sound Machine ...",Yuriy Vitaliyovych Lutsenko (; born 14 Decembe...,22.187500,15.367188,"[0, 1]"
4,Which country is Lake Ewauna associated with?,""" Lake Ewauna "" "" Lake Ewauna "" The Klamath Ri...","Henry Wager Halleck ( January 16 , 1815 – Janu...",19.468750,13.179688,"[0, 1]"
...,...,...,...,...,...,...
245,Which country is Bad Astronaut associated with?,""" Bad Astronaut "" "" Bad Astronaut "" In Bad Ast...",Robert Kingsbury Huntington ( 13 March 1921 – ...,18.375000,18.625000,"[1, 0]"
246,What conflict was Catinat part of?,""" Catinat "" "" Catinat "" Marshal Villeroi repla...","In gridiron football , a triple - threat man i...",21.093750,14.742188,"[0, 1]"
247,Who is the father of Billy?,""" Billy "" "" Billy "" Corral on October 26 , 188...","Tire ( ) is a populous district , as well as t...",19.625000,15.343750,"[0, 1]"
248,Which administrative territorial entity is Dur...,""" Durgada "" "" Durgada "" Durgada has a railway ...",The Denali National Park Improvement Act ( ) i...,19.421875,17.484375,"[0, 1]"


In [6]:
from scipy import stats

def standard_ttest_ppf(n, confidence_level=0.95):
    return stats.t.ppf(q=1-confidence_level, df=n-1, loc=0, scale=1)

df = rdf.copy()
col1, col2 = "score_2", "score_1"
ttest = stats.ttest_rel(df[col1], df[col2])
result = {
    "col1": col1,
    "col2": col2,
    "ttest_stats": ttest[0],
    "ttest_pvalue": ttest[1],
    "ttest_ci_low_stats": ttest.confidence_interval(confidence_level=0.95)[0],
    "ttest_ci_high_stats": ttest.confidence_interval(confidence_level=0.95)[1],
    "ttest_ci_low": np.abs(standard_ttest_ppf(len(df))),
    "ttest_ci_high": np.abs(standard_ttest_ppf(len(df))),
    "standard_ttest_ppf": standard_ttest_ppf(len(df)),
    "acc": (df[col1] > df[col2]).mean(),
}
result

{'col1': 'score_2',
 'col2': 'score_1',
 'ttest_stats': -20.9602704541204,
 'ttest_pvalue': 6.619581919719188e-57,
 'ttest_ci_low_stats': -4.180519796199612,
 'ttest_ci_high_stats': -3.462355203800388,
 'ttest_ci_low': 1.650996151677261,
 'ttest_ci_high': 1.650996151677261,
 'standard_ttest_ppf': -1.650996151677261,
 'acc': 0.076}